In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-mpnet-base-v2")

/home/sudip/Desktop/agent_murdock/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3356.85it/s]


In [2]:
faiss_text = "What is FAISS. FAISS is a local vector library created by Facebook AI Research for fast matrix math."
weaviate_text = "Explaining Vector Databases. Weaviate is an AI-native database that stores both objects and vectors."

In [3]:
v1 = model.encode(weaviate_text).tolist()

In [4]:
v2 = model.encode(faiss_text).tolist()

In [13]:
query_vector = model.encode("Meta's open source embedding search tool").tolist()

In [6]:
import os
import weaviate
from dotenv import load_dotenv
from weaviate.classes.config import Configure, DataType, Property
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

with weaviate.connect_to_embedded(
    headers={"X-Goog-Api-Key": GEMINI_API_KEY}
) as client:
    collection_name = "Articles"

    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
    
    articles = client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(),
        properties=[
            Property(name='title', data_type=DataType.TEXT),
            Property(name='body', data_type=DataType.TEXT),
        ]
    )

    articles.data.insert(
        properties={
            "title": "Explaining Vector Databases",
            "body": "Weaviate is an AI-native database that stores both objects and vectors."
        },
        vector=v1
    )
    articles.data.insert(
        properties={
            "title": "What is FAISS",
            "body": "FAISS is a local vector library created by Facebook AI Research for fast matrix math."
        },
        vector=v2
    )

{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"

In [7]:
from numpy import dot
from numpy.linalg import norm

cos_sim = lambda a, b: dot(a, b) / (norm(a) * norm(b))

In [14]:
print("Similarity to Weaviate article:", cos_sim(query_vector, v1))
print("Similarity to FAISS article:", cos_sim(query_vector, v2))

Similarity to Weaviate article: 0.24015894974673396
Similarity to FAISS article: 0.21078403235477725


# Trial with chunker

In [1]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from app.pipeline.chunker import Chunker

chunker = Chunker(
    doc_list=["../PDF_DOCS/Civil-Rights-Act-Nepal-1955.pdf", "../PDF_DOCS/LandAct1964.pdf", "../PDF_DOCS/NP_National Penal Code Act_EN.pdf", "../PDF_DOCS/The-Labour-Act-2017.pdf"],
    chunk_size=500,
    chunk_overlap=100,
)

/home/sudip/Desktop/agent_murdock/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chunk_values = chunker.chunk_docs()
print(chunk_values)
len(chunk_values)

[Document(metadata={'source': '../PDF_DOCS/Civil-Rights-Act-Nepal-1955.pdf', 'page': 1, 'start_index': 0}, page_content='1 \nCivil Rights Act, 2012 (1955) \n     Date of Authentication   Date of publication in Gaz ette   \n     2012.7.18 (1955 Nov.4)  2012.8.13 (1955 Dec.3)  \n \nAmendments    \n1.  Foreigners Act, 2015 2015.1.16 2015.2.20 \n2.  Some Nepal Acts (Amendment and \nre-organization) Act, 2020 2020.11.16 2020.11.16 \n3. \n Civil Rights (First Amendment) \nAct, 2035 2035.7.8  \n4 \nAdministration of Justice Act, 2048  \n2048.2.16  2048.2.16  \nAct no. 12 of 2012  \nAn Act to provide for Civil Rights to the People  \n Preamble : Whereas, it is expedient to provide for Civil Rig hts to the people and \nprocedure therefor to enforce those Rights, \n Be it enacted and proclaimed by His Majesty the Ki ng. \n1.  Short Title, Extent and Commencement \n: (1) This Act may be called the "Civil \nRights Act, 2012 (1955)". \n(2) This Act shall come into force all over Nepal. \n(3) This A

379

# Test Weaver

In [3]:
from app.pipeline.weaver import Weaver

weaver = Weaver()
weaver.weave_database()
weaver.ingest_chunks(
    chunk_values
)
query = "What is the meaning of Law Documents"
sim_docs = weaver.return_similar_docs(
    query
)

Weaviate is ready.


In [4]:
sim_docs

[Document(metadata={'source': '../PDF_DOCS/The-Labour-Act-2017.pdf', 'page': 4}, page_content='www.lawcommission.gov.np \n4 \n \n(x) "Labour" means a worker or employee or a person emp loyed with \nany job title who performs a physical or intellectual work for the \nemployer.  \n(y) "Labour supplier" means a person or body having o btained the \nlicense pursuant to Chapter 11 to supply labours. \n(z) "Strike" means a situation where labours collectiv ely refuse to \nperform their regular work partially or completely.  \nProvided that a situation where labours are on leav e or \nstopped the work as a result of the occurrence of a n immediate \ndanger or have become unable to attend or perform t he work due to \na situation beyond the control of labours shall not be  considered as \na strike. \n \nChapter-2 \nBasic Provisions Relating to Labours \n3. To remain as minimum standard s: ( 1) This Act shall remain as minimum \nstandards for labours and on matters relating to la bours. \n(2) W

# Rerankers layer

In [5]:
from app.pipeline.ranker import Ranker

ranker = Ranker()
ranked_docs =  ranker.reranked_docs(initial_docs=sim_docs, query=query, num_final_docs=3)
ranked_docs

Loading TransformerRanker model cross-encoder/ms-marco-MiniLM-L-6-v2 (this message can be suppressed by setting verbose=0)
No dtype set
Using dtype torch.float32


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 27096.65it/s]


Loaded model cross-encoder/ms-marco-MiniLM-L-6-v2
Using device cpu.
Using dtype torch.float32.


===> Reranking documents...


[Document(metadata={'source': '../PDF_DOCS/The-Labour-Act-2017.pdf', 'page': 4, 'rerank_score': -6.019613742828369}, page_content='www.lawcommission.gov.np \n4 \n \n(x) "Labour" means a worker or employee or a person emp loyed with \nany job title who performs a physical or intellectual work for the \nemployer.  \n(y) "Labour supplier" means a person or body having o btained the \nlicense pursuant to Chapter 11 to supply labours. \n(z) "Strike" means a situation where labours collectiv ely refuse to \nperform their regular work partially or completely.  \nProvided that a situation where labours are on leav e or \nstopped the work as a result of the occurrence of a n immediate \ndanger or have become unable to attend or perform t he work due to \na situation beyond the control of labours shall not be  considered as \na strike. \n \nChapter-2 \nBasic Provisions Relating to Labours \n3. To remain as minimum standard s: ( 1) This Act shall remain as minimum \nstandards for labours and on m

# Testing Generator

In [6]:
from app.pipeline.generator import Generator

generator = Generator()

final_answer = generator.generate_answer(
    ranked_docs,
    query,
)

final_answer

2026-05-20 08:54:25,688 [INFO] google_genai.models - AFC is enabled with max remote calls: 10.
2026-05-20 08:54:25,689 [DEBUG] httpcore.connection - connect_tcp.started host='generativelanguage.googleapis.com' port=443 local_address=None timeout=30.0 socket_options=None
2026-05-20 08:54:25,760 [DEBUG] httpcore.connection - connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x718fd05b6420>
2026-05-20 08:54:25,760 [DEBUG] httpcore.connection - start_tls.started ssl_context=<ssl.SSLContext object at 0x718fd0591150> server_hostname='generativelanguage.googleapis.com' timeout=30.0
2026-05-20 08:54:25,833 [DEBUG] httpcore.connection - start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x718fd0543650>
2026-05-20 08:54:25,833 [DEBUG] httpcore.http11 - send_request_headers.started request=<Request [b'POST']>
2026-05-20 08:54:25,833 [DEBUG] httpcore.http11 - send_request_headers.complete
2026-05-20 08:54:25,833 [DEBUG] httpcore.http11 - sen

GenResponse(response="THE ANSWER COULDN'T BE FOUND IN THE CONTEXT.", citations={}, token_use=TokenUsage(prompt_token_count=739, total_token_count=765))